# Notebook 5: Optimization — Gradient Descent, Momentum & Adam

---

## Learning Objectives

By the end of this notebook you will be able to:

- Explain the intuition behind gradient descent and its update rule
- Distinguish between batch, mini-batch, and stochastic gradient descent
- Describe how Momentum, RMSProp, and Adam improve upon vanilla SGD
- Implement SGD from scratch in NumPy
- Compare optimizer trajectories visually on contour plots
- Understand learning rate schedules and their effect on convergence

## Prerequisites

- Notebook 03 (Backpropagation) and Notebook 04 (Loss Functions)
- Familiarity with partial derivatives and the chain rule
- Basic PyTorch usage (`torch.optim`, `nn.Module`)

## Table of Contents

1. [Gradient Descent Intuition](#1-gradient-descent-intuition)
2. [Visualizing the Loss Landscape](#2-visualizing-the-loss-landscape)
3. [SGD vs Mini-Batch vs Batch Gradient Descent](#3-sgd-vs-mini-batch-vs-batch-gradient-descent)
4. [Momentum](#4-momentum)
5. [RMSProp](#5-rmsprop)
6. [Adam Optimizer](#6-adam-optimizer)
7. [Learning Rate Schedules](#7-learning-rate-schedules)
8. [Code: SGD from Scratch in NumPy](#8-code-sgd-from-scratch-in-numpy)
9. [Code: Compare Optimizers in PyTorch](#9-code-compare-optimizers-in-pytorch)
10. [Code: Visualize Optimizer Trajectories](#10-code-visualize-optimizer-trajectories)
11. [Exercise: Tune Learning Rate and Observe Effects](#11-exercise)
12. [Common Mistakes & Debugging Tips](#12-common-mistakes--debugging-tips)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from matplotlib import cm

np.random.seed(42)
torch.manual_seed(42)

%matplotlib inline
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['figure.dpi'] = 100

---
## 1. Gradient Descent Intuition

Gradient descent is the workhorse of deep learning optimization. The core idea is simple:

- Compute the gradient (direction of steepest *increase*) of the loss with respect to the parameters.
- Move the parameters in the *opposite* direction to reduce the loss.

### Update Rule

$$w \leftarrow w - \eta \nabla L(w)$$

where:
- $w$ = model parameters (weights)
- $\eta$ = learning rate (step size)
- $\nabla L(w)$ = gradient of the loss with respect to $w$

**Key points:**
- A *large* learning rate risks overshooting the minimum.
- A *small* learning rate converges slowly and may get stuck in local minima.
- The gradient magnitude naturally shrinks near a minimum (for convex losses).

---
## 2. Visualizing the Loss Landscape

Let us visualize gradient descent on a 2D loss surface. We use a simple quadratic:

$$L(w_1, w_2) = w_1^2 + 5\,w_2^2$$

This surface is bowl-shaped but elongated along $w_2$, making optimization harder for vanilla SGD.

In [ ]:
def loss_fn_2d(w1, w2):
    """Elongated quadratic: L = w1^2 + 5*w2^2."""
    return w1**2 + 5 * w2**2

def grad_fn_2d(w1, w2):
    """Gradient of L = w1^2 + 5*w2^2."""
    return np.array([2 * w1, 10 * w2])

# Create grid for contour plot
w1_range = np.linspace(-5, 5, 200)
w2_range = np.linspace(-5, 5, 200)
W1, W2 = np.meshgrid(w1_range, w2_range)
L = loss_fn_2d(W1, W2)

fig, ax = plt.subplots(1, 1, figsize=(8, 6))
contour = ax.contour(W1, W2, L, levels=20, cmap='viridis')
ax.clabel(contour, inline=True, fontsize=8)

# Draw gradient arrows at a few sample points
sample_points = [(-4, 3), (3, -2), (-2, -3), (2, 4)]
for (pw1, pw2) in sample_points:
    g = grad_fn_2d(pw1, pw2)
    # Normalize for display, point in -gradient direction (descent)
    g_norm = g / (np.linalg.norm(g) + 1e-8)
    ax.annotate('', xy=(pw1 - 0.8 * g_norm[0], pw2 - 0.8 * g_norm[1]),
                xytext=(pw1, pw2),
                arrowprops=dict(arrowstyle='->', color='red', lw=2))
    ax.plot(pw1, pw2, 'ro', markersize=5)

ax.set_xlabel('$w_1$')
ax.set_ylabel('$w_2$')
ax.set_title('Loss Landscape $L = w_1^2 + 5w_2^2$ with Gradient Arrows (descent direction)')
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

The red arrows point in the *negative gradient* direction (the direction that reduces the loss). Notice how the gradient direction does not always point toward the minimum on elongated surfaces.

---
## 3. SGD vs Mini-Batch vs Batch Gradient Descent

| Variant | Batch size | Gradient estimate | Speed per update | Convergence |
|---|---|---|---|---|
| **Batch GD** | Entire dataset | Exact | Slow | Smooth |
| **Stochastic GD** | 1 sample | Very noisy | Fast | Noisy, oscillates |
| **Mini-batch GD** | $B$ samples (e.g., 32–256) | Moderate noise | Good trade-off | Smooth-ish |

**Practical notes:**
- Mini-batch GD is used almost universally in practice (batch sizes 32–512).
- The noise in SGD/mini-batch can actually *help* escape shallow local minima.
- Larger batch sizes give more stable gradients but consume more memory.

### Gradient Update Formulas

- **Batch GD:** $w \leftarrow w - \eta \frac{1}{N} \sum_{i=1}^{N} \nabla L_i(w)$
- **SGD:** $w \leftarrow w - \eta \nabla L_i(w)$ (single random sample $i$)
- **Mini-batch:** $w \leftarrow w - \eta \frac{1}{B} \sum_{i \in \mathcal{B}} \nabla L_i(w)$ (random subset $\mathcal{B}$)

---
## 4. Momentum

Momentum adds a "velocity" term that accumulates past gradients, helping the optimizer:
- **Accelerate** in directions with consistent gradient.
- **Dampen** oscillations in directions with sign-changing gradient.

### Update Rules

$$v_t = \beta\, v_{t-1} + \nabla L(w_t)$$
$$w_{t+1} = w_t - \eta\, v_t$$

where $\beta \in [0, 1)$ is the momentum coefficient (typically 0.9).

**Intuition:** Think of a ball rolling down a hill. On a flat surface it keeps rolling because of accumulated velocity. In a narrow ravine it oscillates less because opposing oscillations cancel out in the velocity accumulator.

---
## 5. RMSProp

RMSProp adapts the learning rate *per parameter* by dividing by the running average of squared gradients.

### Update Rules

$$s_t = \beta\, s_{t-1} + (1 - \beta)(\nabla L)^2$$
$$w \leftarrow w - \frac{\eta}{\sqrt{s_t} + \epsilon} \nabla L$$

where $\beta \approx 0.99$ and $\epsilon \approx 10^{-8}$.

**Key insight:** Parameters with large gradients get a smaller effective learning rate. Parameters with small gradients get a larger effective learning rate. This helps navigate loss surfaces with very different curvatures along different dimensions.

---
## 6. Adam Optimizer

Adam (Adaptive Moment Estimation) combines the ideas of Momentum and RMSProp.

### Update Rules

**First moment (mean of gradients, like momentum):**
$$m_t = \beta_1\, m_{t-1} + (1 - \beta_1) \nabla L$$

**Second moment (mean of squared gradients, like RMSProp):**
$$v_t = \beta_2\, v_{t-1} + (1 - \beta_2) (\nabla L)^2$$

**Bias correction** (important in early steps when $m$ and $v$ are biased toward zero):
$$\hat{m}_t = \frac{m_t}{1 - \beta_1^t}, \qquad \hat{v}_t = \frac{v_t}{1 - \beta_2^t}$$

**Parameter update:**
$$w \leftarrow w - \frac{\eta}{\sqrt{\hat{v}_t} + \epsilon} \hat{m}_t$$

**Default hyperparameters:** $\beta_1 = 0.9$, $\beta_2 = 0.999$, $\epsilon = 10^{-8}$, $\eta = 0.001$.

**Why Adam is popular:**
- Works well out-of-the-box with default hyperparameters.
- Adapts learning rate per parameter.
- Handles sparse gradients well.
- Fast convergence in practice.

---
## 7. Learning Rate Schedules

Instead of using a fixed learning rate, schedules gradually change $\eta$ during training.

| Schedule | Idea |
|---|---|
| **StepLR** | Multiply $\eta$ by a factor every $k$ epochs |
| **CosineAnnealing** | Smoothly decay $\eta$ following a cosine curve |
| **ReduceOnPlateau** | Reduce $\eta$ when validation loss stops improving |

### StepLR
$$\eta_t = \eta_0 \cdot \gamma^{\lfloor t / \text{step\_size} \rfloor}$$

### CosineAnnealing
$$\eta_t = \eta_{\min} + \frac{1}{2}(\eta_{\max} - \eta_{\min})\left(1 + \cos\left(\frac{t}{T_{\max}} \pi\right)\right)$$

In [ ]:
# Visualize learning rate schedules
epochs = 100
lr_init = 0.1

# StepLR schedule
step_size, gamma = 30, 0.1
step_lrs = [lr_init * gamma ** (e // step_size) for e in range(epochs)]

# CosineAnnealing schedule
eta_min = 1e-4
cosine_lrs = [
    eta_min + 0.5 * (lr_init - eta_min) * (1 + np.cos(np.pi * e / epochs))
    for e in range(epochs)
]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(step_lrs, 'b-', linewidth=2)
axes[0].set_title('StepLR Schedule')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Learning Rate')
axes[0].set_ylim(bottom=0)
axes[0].grid(True, alpha=0.3)

axes[1].plot(cosine_lrs, 'r-', linewidth=2)
axes[1].set_title('CosineAnnealing Schedule')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Learning Rate')
axes[1].set_ylim(bottom=0)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 8. Code: SGD from Scratch in NumPy

We implement SGD (with optional momentum) on a simple quadratic loss:

$$L(w) = \frac{1}{2} w^T A w$$

where $A = \text{diag}(1, 10)$ creates an elongated bowl.

In [ ]:
def sgd_numpy(grad_fn, w_init, lr=0.01, momentum=0.0, n_steps=50):
    """
    SGD with optional momentum, implemented from scratch.
    
    Args:
        grad_fn: function that returns the gradient at a point
        w_init: initial parameter values (numpy array)
        lr: learning rate
        momentum: momentum coefficient (0 = vanilla SGD)
        n_steps: number of update steps
    
    Returns:
        trajectory: list of (w1, w2) positions visited
    """
    w = np.array(w_init, dtype=float)
    v = np.zeros_like(w)  # velocity
    trajectory = [w.copy()]
    
    for _ in range(n_steps):
        g = grad_fn(w[0], w[1])
        v = momentum * v + g
        w = w - lr * v
        trajectory.append(w.copy())
    
    return np.array(trajectory)

# Run vanilla SGD
w0 = np.array([-4.0, 4.0])
traj_sgd = sgd_numpy(grad_fn_2d, w0, lr=0.05, momentum=0.0, n_steps=50)
traj_mom = sgd_numpy(grad_fn_2d, w0, lr=0.05, momentum=0.9, n_steps=50)

print(f"Vanilla SGD final position: w = [{traj_sgd[-1, 0]:.4f}, {traj_sgd[-1, 1]:.4f}]")
print(f"Momentum SGD final position: w = [{traj_mom[-1, 0]:.4f}, {traj_mom[-1, 1]:.4f}]")
print(f"\nVanilla SGD steps to reach ||w|| < 0.1: ", end="")
for i, w in enumerate(traj_sgd):
    if np.linalg.norm(w) < 0.1:
        print(i)
        break
else:
    print("Did not converge")

In [ ]:
# Plot SGD vs Momentum trajectories on the contour plot
fig, ax = plt.subplots(1, 1, figsize=(8, 6))
ax.contour(W1, W2, L, levels=20, cmap='viridis', alpha=0.6)

ax.plot(traj_sgd[:, 0], traj_sgd[:, 1], 'bo-', markersize=3, linewidth=1.5, label='Vanilla SGD')
ax.plot(traj_mom[:, 0], traj_mom[:, 1], 'rs-', markersize=3, linewidth=1.5, label='SGD + Momentum')
ax.plot(0, 0, 'k*', markersize=15, label='Minimum')
ax.plot(w0[0], w0[1], 'gD', markersize=10, label='Start')

ax.set_xlabel('$w_1$')
ax.set_ylabel('$w_2$')
ax.set_title('Vanilla SGD vs SGD + Momentum on $L = w_1^2 + 5w_2^2$')
ax.legend()
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

---
## 9. Code: Compare Optimizers in PyTorch

We compare SGD, SGD+Momentum, RMSProp, and Adam on a toy regression problem.

In [ ]:
torch.manual_seed(42)
np.random.seed(42)

# Toy dataset: noisy sine wave
X = torch.linspace(-3, 3, 200).unsqueeze(1)
y = torch.sin(X) + 0.1 * torch.randn_like(X)

def make_model():
    """Create a small MLP for regression."""
    torch.manual_seed(42)
    return nn.Sequential(
        nn.Linear(1, 32),
        nn.ReLU(),
        nn.Linear(32, 32),
        nn.ReLU(),
        nn.Linear(32, 1)
    )

def train_with_optimizer(optimizer_class, lr, model=None, epochs=300, **kwargs):
    """Train and return loss history."""
    if model is None:
        model = make_model()
    optimizer = optimizer_class(model.parameters(), lr=lr, **kwargs)
    loss_fn = nn.MSELoss()
    losses = []
    
    for epoch in range(epochs):
        pred = model(X)
        loss = loss_fn(pred, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    
    return losses

# Train with different optimizers
losses_sgd = train_with_optimizer(optim.SGD, lr=0.01)
losses_mom = train_with_optimizer(optim.SGD, lr=0.01, momentum=0.9)
losses_rms = train_with_optimizer(optim.RMSprop, lr=0.001)
losses_adam = train_with_optimizer(optim.Adam, lr=0.001)

# Plot
plt.figure(figsize=(10, 5))
plt.plot(losses_sgd, label='SGD (lr=0.01)', alpha=0.8)
plt.plot(losses_mom, label='SGD+Momentum (lr=0.01, beta=0.9)', alpha=0.8)
plt.plot(losses_rms, label='RMSProp (lr=0.001)', alpha=0.8)
plt.plot(losses_adam, label='Adam (lr=0.001)', alpha=0.8)
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Optimizer Comparison on Toy Regression')
plt.legend()
plt.yscale('log')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 10. Code: Visualize Optimizer Trajectories

We implement Adam from scratch in NumPy and compare all four optimizer trajectories on the 2D contour plot.

In [ ]:
def adam_numpy(grad_fn, w_init, lr=0.05, beta1=0.9, beta2=0.999, eps=1e-8, n_steps=50):
    """Adam optimizer implemented from scratch in NumPy."""
    w = np.array(w_init, dtype=float)
    m = np.zeros_like(w)  # first moment
    v = np.zeros_like(w)  # second moment
    trajectory = [w.copy()]
    
    for t in range(1, n_steps + 1):
        g = grad_fn(w[0], w[1])
        m = beta1 * m + (1 - beta1) * g
        v = beta2 * v + (1 - beta2) * g**2
        m_hat = m / (1 - beta1**t)  # bias correction
        v_hat = v / (1 - beta2**t)
        w = w - lr * m_hat / (np.sqrt(v_hat) + eps)
        trajectory.append(w.copy())
    
    return np.array(trajectory)

def rmsprop_numpy(grad_fn, w_init, lr=0.05, beta=0.99, eps=1e-8, n_steps=50):
    """RMSProp optimizer implemented from scratch in NumPy."""
    w = np.array(w_init, dtype=float)
    s = np.zeros_like(w)
    trajectory = [w.copy()]
    
    for _ in range(n_steps):
        g = grad_fn(w[0], w[1])
        s = beta * s + (1 - beta) * g**2
        w = w - lr * g / (np.sqrt(s) + eps)
        trajectory.append(w.copy())
    
    return np.array(trajectory)

# Run all optimizers from the same starting point
w0 = np.array([-4.0, 4.0])
n_steps = 80

traj_sgd = sgd_numpy(grad_fn_2d, w0, lr=0.05, momentum=0.0, n_steps=n_steps)
traj_mom = sgd_numpy(grad_fn_2d, w0, lr=0.05, momentum=0.9, n_steps=n_steps)
traj_rms = rmsprop_numpy(grad_fn_2d, w0, lr=0.1, n_steps=n_steps)
traj_adam = adam_numpy(grad_fn_2d, w0, lr=0.3, n_steps=n_steps)

# Plot all trajectories
fig, ax = plt.subplots(1, 1, figsize=(10, 7))
ax.contour(W1, W2, L, levels=30, cmap='viridis', alpha=0.5)

for traj, label, marker, color in [
    (traj_sgd, 'Vanilla SGD', 'o', 'blue'),
    (traj_mom, 'SGD + Momentum', 's', 'red'),
    (traj_rms, 'RMSProp', '^', 'green'),
    (traj_adam, 'Adam', 'D', 'purple'),
]:
    ax.plot(traj[:, 0], traj[:, 1], marker=marker, color=color,
            markersize=3, linewidth=1.2, label=label, alpha=0.8)

ax.plot(0, 0, 'k*', markersize=15, label='Minimum')
ax.plot(w0[0], w0[1], 'kD', markersize=10, label='Start')
ax.set_xlabel('$w_1$')
ax.set_ylabel('$w_2$')
ax.set_title('Optimizer Trajectories on $L = w_1^2 + 5w_2^2$')
ax.legend(loc='upper right')
ax.set_xlim(-5, 5)
ax.set_ylim(-5, 5)
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

**Observations:**
- Vanilla SGD oscillates in the high-curvature $w_2$ direction.
- Momentum dampens oscillations and accelerates toward the minimum.
- RMSProp and Adam adapt their step size per dimension, handling the elongated bowl more efficiently.
- Adam typically converges fastest because it combines the benefits of both momentum and adaptive learning rates.

---
<a id='11-exercise'></a>
## 11. Exercise: Tune Learning Rate and Observe Effects

**Task:** Experiment with different learning rates for vanilla SGD on the quadratic loss and observe convergence behavior.

1. Try `lr = 0.001` (too small), `lr = 0.05` (reasonable), `lr = 0.19` (near the stability limit for this problem).
2. Plot the trajectories and note how many steps each takes.
3. What happens if `lr = 0.21`? (Hint: the loss *diverges* when $\eta > 1/\lambda_{\max}$, where $\lambda_{\max} = 10$ here.)

In [ ]:
# ============================================================
# EXERCISE: Uncomment and complete the code below
# ============================================================

# learning_rates = [0.001, 0.05, 0.19]
# w0 = np.array([-4.0, 4.0])
#
# fig, ax = plt.subplots(1, 1, figsize=(10, 7))
# ax.contour(W1, W2, L, levels=20, cmap='viridis', alpha=0.5)
#
# for lr in learning_rates:
#     traj = sgd_numpy(grad_fn_2d, w0, lr=lr, momentum=0.0, n_steps=100)
#     ax.plot(traj[:, 0], traj[:, 1], 'o-', markersize=2, label=f'lr={lr}')
#
# ax.plot(0, 0, 'k*', markersize=15)
# ax.set_xlabel('$w_1$')
# ax.set_ylabel('$w_2$')
# ax.set_title('Effect of Learning Rate on SGD')
# ax.legend()
# ax.set_aspect('equal')
# plt.tight_layout()
# plt.show()
#
# # BONUS: Try lr=0.21 and see what happens
# # traj_diverge = sgd_numpy(grad_fn_2d, w0, lr=0.21, n_steps=20)
# # print("Diverging trajectory final point:", traj_diverge[-1])

### Solution

In [ ]:
learning_rates = [0.001, 0.05, 0.19]
w0 = np.array([-4.0, 4.0])

fig, ax = plt.subplots(1, 1, figsize=(10, 7))
ax.contour(W1, W2, L, levels=20, cmap='viridis', alpha=0.5)

for lr in learning_rates:
    traj = sgd_numpy(grad_fn_2d, w0, lr=lr, momentum=0.0, n_steps=100)
    ax.plot(traj[:, 0], traj[:, 1], 'o-', markersize=2, label=f'lr={lr}')

ax.plot(0, 0, 'k*', markersize=15)
ax.set_xlabel('$w_1$')
ax.set_ylabel('$w_2$')
ax.set_title('Effect of Learning Rate on SGD')
ax.legend()
ax.set_xlim(-6, 6)
ax.set_ylim(-6, 6)
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

# Diverging case
traj_diverge = sgd_numpy(grad_fn_2d, w0, lr=0.21, n_steps=20)
print(f"lr=0.21 diverges! Final point: [{traj_diverge[-1, 0]:.2f}, {traj_diverge[-1, 1]:.2f}]")
print(f"The stability limit is lr < 1/lambda_max = 1/10 = 0.1 for the w2 direction.")
print(f"lr=0.19 works because 0.19 < 1/5 = 0.2 for the FULL gradient (eigenvalue is 10, lr < 2/10 = 0.2).")

---
## 12. Common Mistakes & Debugging Tips

| Mistake | Symptom | Fix |
|---|---|---|
| Learning rate too high | Loss explodes (NaN or inf) | Reduce lr by 10x; use lr finder |
| Learning rate too low | Loss decreases very slowly | Increase lr; use a schedule |
| Forgetting `optimizer.zero_grad()` | Gradients accumulate across batches; loss behaves erratically | Always call before `.backward()` |
| Not using bias correction in Adam | Slow start, biased updates in early steps | Use PyTorch's built-in `Adam` which handles this |
| Using SGD without momentum | Slow convergence on ill-conditioned problems | Add `momentum=0.9` |
| Fixed LR for entire training | Suboptimal final loss | Add a scheduler (CosineAnnealing is a good default) |

**Debugging checklist:**
1. Print the loss every epoch — it should generally decrease.
2. If loss is NaN, check for division by zero, log of non-positive values, or lr too high.
3. Monitor gradient norms: `torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)`.
4. Try Adam first as a baseline — it is forgiving and works well without much tuning.
5. If Adam overfits, switch to SGD + momentum with a cosine schedule (common in vision tasks).